# Task
$\mathcal H _\infty$ controller design and analysis into Python.

## Setup

### Subtask:
Install necessary Python libraries for control system design and visualization.


**Remark**:
Install the required libraries for control system design and plotting.



The `hinfsyn` function requires the `slycot` library, which is not installed. I need to install `slycot`.



In [ ]:
#!apt-get install gfortran cmake --fix-missing
#!apt-get install libblas-dev liblapack-dev

import os
os.environ['BLA_VENDOR']="Generic"
!echo Using $$BLA_VENDOR BLAS implementation

!pip install slycot
!pip install control


Using Generic BLAS implementation
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for slycot: filename=slycot-0.6.0-cp312-cp312-linux_x86_64.whl size=2034911 sha256=d104066f92cd8cfca0de2dcf666381a965247e1aa27150cdf42e51722920079b
  Stored in directory: /root/.cache/pip/wheels/43/4b/67/3eb0ae4ff6eeb3939abdfa66fa4640d285ecc35ad00de1d19d
Successfully built slycot
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 12.9 MB/s eta 0:00:00


## Analyze Closed-Loop System (Continuous-Time)

### Subtask:
Analyze the continuous-time closed-loop system, including calculating the direct loop function, the closed-loop system, and verifying performance and robustness using Bode plots.

**Reasoning**:
Calculate the direct loop function `L`, the closed-loop system `sysnom_CL`, and generate Bode plots of the sensitivity function `S` and the closed-loop transfer function `T` to compare against the inverse weighting functions `1/W1` and `1/W3`, respectively. Also, plot the Bode magnitude of `S*K1` to compare against `1/W2`.

In [1]:
import matplotlib.pyplot as plt
import slycot
import control as ct
import numpy as np
from control.robust import hinfsyn

A = [[0, 1], [-3, -4]]
B = [[0], [1]]
C = [[1, 0]]
D = [[0]]

sys = ct.ss(A, B, C, D)


M1 = 1
A1 = 0.001
Wb1 = 10000

M3 = 0.005
A3 = 1
Wb3 = 1000*np.pi*180

Wp = 1 * ct.tf([1], [1])
W1 = ct.tf([1/M1, Wb1], [1, A1*Wb1])
W2 = 0.00005 * ct.tf([1], [1])
W3 = ct.tf([1/M3, Wb3], [1, A3*Wb3])
Wn = 0.01 * ct.tf([1], [1])


# Create the generalized plant
G = ct.augw(sys, W1,W2,W3)

#print("Generalized Plant P (constructed with interconnect):")
#display(G)

# Now, design the H-infinity controller with the correct P, nmeas, ncon
# nmeas: number of measurements (output 'y_meas' = input_W1_sum), which is 1
# ncon: number of control inputs ('u'), which is 1
nmeas = 1
ncon = 1

# Need to import hinfsyn from control.robust


# Handle potential extra return values
results = hinfsyn(G, nmeas=nmeas, ncon=ncon)
print(f"hinfsyn returned {len(results)} values.")
K1, cl, gam1 = results[:3] # Unpack the first 3 values

print("\nAchieved gamma value (from hinfsyn):")
display(gam1)

print("\nDesigned Controller K1 (State-Space):")
#display(K1)

# Calculate the loop transfer function L
L = sys* K1

print("\nLoop Transfer Function L:")
#display(L)

# Calculate the closed-loop system sysnom_CL = feedback(L, 1)
sysnom_CL = ct.feedback(L, 1)

print("\nNominal Closed-Loop System (sysnom_CL):")
#display(sysnom_CL)

# Plot the step response of sysnom_CL
plt.figure(1)
ct.step_response(sysnom_CL, T=10).plot()
plt.title('Step Response of Nominal Closed-Loop System')
plt.grid(True)


# Comprobacion W1 (Sensitivity function S vs 1/W1)
# S = feedback(1,L)
S = ct.feedback(1, L)

plt.figure(2)
ct.frequency_response([S, 1/W1]).plot(plot_phase=False, overlay_inputs=True, overlay_outputs=True)
plt.title('Bode Magnitude of Sensitivity Function S and 1/W1')
plt.grid(True)
# plt.show() # Removed plt.show()

# Comprobacion W2 (|S*K1| vs 1/W2)
# This check is often for control effort limitation or noise rejection at the output.
# |S*K| represents the transfer function from output disturbance to control input.
SK1 = S * K1

plt.figure(3)
ct.frequency_response([SK1, 1/W2]).plot(plot_phase=False, overlay_inputs=True, overlay_outputs=True)
plt.title('Bode Magnitude of |S*K1| and 1/W2')
plt.grid(True)
# plt.show() # Removed plt.show()

# Comprobacion W3 (Closed-loop transfer function T vs 1/W3)
# T = feedback(L,1) - this is the same as sysnom_CL
T = sysnom_CL # Or ct.feedback(L, 1)

plt.figure(4)
ct.frequency_response([T, 1/W3]).plot(plot_phase=False, overlay_inputs=True, overlay_outputs=True)
plt.title('Bode Magnitude of Closed-Loop Transfer Function T and 1/W3')
plt.grid(True)
# plt.show() # Removed plt.show()

plt.figure(5)
ct.step_response(SK1, T=500).plot()
plt.title('Step Response of Nominal CONTROL ACTION in Closed-Loop')
plt.grid(True)

ModuleNotFoundError: No module named 'slycot'

## Discretize Controller

### Subtask:
Discretize the continuous-time H-infinity controller using a Python equivalent of the MATLAB `c2d` function.

**Remarmk**:
Convert the continuous-time controller `K1` to a discrete-time controller `Kd` using the `c2d` function with the specified sampling time (0.0001 in the MATLAB code).

In [ ]:
# Discretize the continuous-time controller K1
# Use the sampling time from the MATLAB code: 0.0001
Ts = 0.0001
Kd = ct.c2d(K1, Ts)

print("\nDiscrete-Time Controller Kd (State-Space):")
display(Kd)


Discrete-Time Controller Kd (State-Space):


StateSpace(
array([[ 9.99000502e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 7.47382137e-03, -7.05653935e-11,  1.59506986e-02,
         3.34348494e-14],
       [ 4.29853606e-01, -3.98296894e-09,  9.00315779e-01,
         1.88718521e-12],
       [ 4.07456730e+03,  4.18250425e-06, -9.45419017e+02,
        -1.98172777e-09]]),
array([[9.99500167e-03],
       [3.73574815e-05],
       [2.18725402e-03],
       [4.29853606e+01]]),
array([[ 2.16134898e+15, -1.19126270e+09, -5.00967776e+14,
        -4.77068058e+11]]),
array([[0.]]),
dt=0.0001,
name='sys[29]$sampled', states=4, outputs=1, inputs=1)